# RNA3D thesis: TBM + DRfold2 + Geometry v2

Native-blind runtime inference for the public notebook run and hidden competition rerun. GeoFuse is deliberately excluded.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time

import pandas as pd
import torch

INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
runtime_priors = list(INPUT.rglob('geofuse_geometry_v2_priors.json'))
runtime_archives = list(INPUT.rglob('rna3d_hybrid_runtime.tar.gz'))
if len(runtime_priors) == 1:
    RUNTIME = runtime_priors[0].parent
elif len(runtime_archives) == 1:
    RUNTIME = WORKING / 'rna3d_runtime'
    if RUNTIME.exists():
        shutil.rmtree(RUNTIME)
    RUNTIME.mkdir(parents=True)
    with tarfile.open(runtime_archives[0], 'r:gz') as archive:
        archive.extractall(RUNTIME)
else:
    raise FileNotFoundError(f'hybrid runtime is missing: priors={runtime_priors}, archives={runtime_archives}')

bundle_manifests = list(INPUT.rglob('bundle_manifest.json'))
bundle_archives = list(INPUT.rglob('rna3d_bundle.tar.gz'))
if bundle_manifests:
    BUNDLE = bundle_manifests[0].parent
elif bundle_archives:
    BUNDLE = WORKING / 'rna3d_bundle'
    BUNDLE.mkdir(parents=True, exist_ok=True)
    with tarfile.open(bundle_archives[0], 'r:gz') as archive:
        archive.extractall(BUNDLE)
else:
    raise FileNotFoundError('TBM artifact bundle is missing')

competition_roots = [
    path.parent for path in INPUT.rglob('test_sequences.csv')
    if (path.parent / 'sample_submission.csv').is_file()
]
if len(competition_roots) != 1:
    raise FileNotFoundError(f'expected one competition input, found {competition_roots}')
COMPETITION = competition_roots[0]
SITE_PACKAGES = Path('/kaggle/temp/rna3d_site_packages')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps',
    '--find-links', str(RUNTIME / 'wheels'), '--target', str(SITE_PACKAGES), 'biopython'
], check=True)
sys.path.insert(0, str(SITE_PACKAGES))
sys.path.insert(0, str(RUNTIME))
sys.path.insert(0, str(RUNTIME / 'src'))
os.environ['RNA3D_PROCESSED'] = str(BUNDLE)
os.environ['RNA3D_CACHE'] = str(BUNDLE)
os.environ['RNA3D_MMSEQS'] = str(BUNDLE / 'bin' / 'mmseqs')
old_ld_library_path = os.environ.get('LD_LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = f'{BUNDLE / "lib"}:{old_ld_library_path}' if old_ld_library_path else str(BUNDLE / 'lib')
print(f'cuda={torch.cuda.is_available()} gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')


In [ ]:
from kaggle.hybrid_inference import run_hybrid_inference

test = pd.read_csv(COMPETITION / 'test_sequences.csv', dtype=str)
sample = pd.read_csv(COMPETITION / 'sample_submission.csv')
submission, manifest = run_hybrid_inference(
    test,
    BUNDLE,
    RUNTIME,
    INPUT,
    work_dir=Path('/kaggle/temp/rna3d_hybrid_work'),
    sample_submission=sample,
    geometry_steps=300,
    drfold_max_len=600,
    drfold_deadline_seconds=6.5 * 60 * 60,
)
output = WORKING / 'submission.csv'
submission.to_csv(output, index=False)
(WORKING / 'hybrid_inference_manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n')
assert submission.shape == sample.shape
assert submission['ID'].tolist() == sample['ID'].tolist()
assert not submission.isna().any().any()
print(f'wrote {output}; targets={len(test)}; rows={len(submission)}; runtime={manifest["elapsed_seconds"]}s')


In [ ]:
submission.head(3)